RAG with LangChain

In [15]:
%pip install langchain-google-genai langchain langchain_core langchain_community langchain-chroma chromadb sentence-transformers dotenv pypdf langchain-huggingface -q
print("Dependencies installed succesfully!")

Note: you may need to restart the kernel to use updated packages.
Dependencies installed succesfully!


In [16]:
#loading the documents
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("air-rag.pdf")
docs = loader.load()

docs[0].metadata
print("Docs loaded.")

Ignoring wrong pointing object 85 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)
Ignoring wrong pointing object 133 0 (offset 0)
Ignoring wrong pointing object 178 0 (offset 0)
Ignoring wrong pointing object 346 0 (offset 0)
Ignoring wrong pointing object 347 0 (offset 0)


Docs loaded.


In [17]:
#split into chunks
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = splitter.split_documents(docs)
print("Length of chunks : ",len(chunks))
print("Docs split into chunks.")

Length of chunks :  104
Docs split into chunks.


In [ ]:
#load embedding model form hugging face
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()

hf_token = os.getenv("HUGGINGFACE_TOKEN")
if not hf_token:
    raise ValueError("Missing Hugging Face token. Set your Huggingface Token in .env file.")

login(token = hf_token)


In [19]:
#creating embeddings
from langchain_huggingface import HuggingFaceEmbeddings

loading_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded")

embeddings = loading_model.embed_documents([c.page_content for c in chunks])
print("Total embeddings : ",len(embeddings))
print("Dimension of embeddings : ",len(embeddings[0]))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8934.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Total embeddings :  104
Dimension of embeddings :  384


In [20]:
#storing in vector db
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents = chunks,
    collection_name = "docs_collection",
    embedding = loading_model,
    persist_directory = "./chroma_db"
)
print("Vector Store created.")

Vector Store created.


In [21]:
#retrieving from vector db - improved with higher k and MMR
retriever = vector_store.as_retriever(
    search_type = 'mmr',  # Maximal Marginal Relevance for diversity
    search_kwargs = {
        'k': 10,           #retrieve more chunks for better coverage
        'fetch_k': 50,     #fetch 50 docs first, then select top 10 diverse ones
        'lambda_mult': 0.5 #balance between similarity and diversity (0.5 = balanced)
    }
)
print("Retriever loaded with MMR search.")

Retriever loaded with MMR search.


In [22]:
#prompt for llm - improved with metadata awareness and source citations
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context given below.
If you don't know the answer, say you don't know.

IMPORTANT: The context includes source markers like [Source 1: Page X, Title: Y].
Cite the relevant sources in your answer using [Source N] format after each claim.

Context : {context}
Question : {question}
""")

In [23]:
from dotenv import load_dotenv
import os

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [24]:
#loading gemini model for answer generation
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = "gemini-flash-latest",
    temperature = 0
)

print("LLM loaded.")

LLM loaded.


In [25]:
#creating the RAG chain - IMPROVED with metadata in context and source citations
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

#format documents to string - INCLUDE METADATA with source markers
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs):
        #include metadata (title, author, page) with each chunk
        meta = doc.metadata
        header = f"[Source {i+1}: Page {meta.get('page', 'N/A')}, Title: {meta.get('title', 'N/A')}]"
        formatted.append(f"{header}\n{doc.page_content}")
    return "\n\n".join(formatted)

#helper function to extract unique sources
def get_sources(docs):
    sources = []
    seen = set()
    for doc in docs:
        meta = doc.metadata
        source_key = f"Page {meta.get('page', 'N/A')}"
        if source_key not in seen:
            seen.add(source_key)
            sources.append({
                "page": meta.get('page', 'N/A'),
                "title": meta.get('title', 'N/A')
            })
    return sources

#wrapper function to get answer with sources
def ask_with_sources(question):
    retrieved_docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)
    sources = get_sources(retrieved_docs)
    return {"answer": answer, "sources": sources}

#building the chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created.")

RAG chain created.


In [26]:
#debug: inspect retrieved chunks for a sample question
test_query = "What is the title of the paper?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: {test_query}\n")
print(f"Number of chunks retrieved: {len(retrieved_docs)}\n")
print("=" * 80)

for i, doc in enumerate(retrieved_docs):
    print(f"\n[Chunk {i+1}] Page: {doc.metadata.get('page', 'N/A')}, Title: {doc.metadata.get('title', 'N/A')}")
    print(f"Content preview: {doc.page_content[:200]}...")
    print("-" * 80)

Query: What is the title of the paper?

Number of chunks retrieved: 10


[Chunk 1] Page: 10, Title: Adaptive iterative retrieval for enhanced retrieval-augmented generation
Content preview: Iter-2 Sentences
s_1 Kate Millett Katherine Murray Millett (September 14, 1934 – September 6, 2017) was an
American feminist writer, educator, artist, and activist.
 
s_2 She attended Oxford Univers...
--------------------------------------------------------------------------------

[Chunk 2] Page: 13, Title: Adaptive iterative retrieval for enhanced retrieval-augmented generation
Content preview: areas include natural language processing (NLP) and rein­ 
forcement learning/machine learning (RL/ML). He has pub­ 
lished more than 60 papers in leading AI and NLP venues 
such as NeurIPS, ACL, an...
--------------------------------------------------------------------------------

[Chunk 3] Page: 11, Title: Adaptive iterative retrieval for enhanced retrieval-augmented generation
Content preview: s_1 

In [27]:
#q&a demo with source citations
questions = [
    "What is the title of the paper?",
    "What is AIR-RAG?",
    "What are the main contributions of this research?",
    "Which university did Kate Millett attend?",
    "What is the main advantage of iterative retrieval?"
]

for q in questions:
    print(f"Question: {q}")
    result = ask_with_sources(q)
    print(f"Answer: {result['answer']}")
    print("\nSources:")
    for src in result['sources']:
        print(f"  • Page {src['page']}: {src['title']}")
    print("-" * 80)

Question: What is the title of the paper?
Answer: The title of the paper is "Adaptive iterative retrieval for enhanced retrieval-augmented generation" [Source 1][Source 2][Source 3][Source 4][Source 5][Source 6][Source 7][Source 8][Source 9][Source 10].

Sources:
  • Page 10: Adaptive iterative retrieval for enhanced retrieval-augmented generation
  • Page 13: Adaptive iterative retrieval for enhanced retrieval-augmented generation
  • Page 11: Adaptive iterative retrieval for enhanced retrieval-augmented generation
  • Page 0: Adaptive iterative retrieval for enhanced retrieval-augmented generation
  • Page 7: Adaptive iterative retrieval for enhanced retrieval-augmented generation
  • Page 9: Adaptive iterative retrieval for enhanced retrieval-augmented generation
--------------------------------------------------------------------------------
Question: What is AIR-RAG?
Answer: AIR-RAG is an adaptive, iterative retrieval framework designed to optimize the retrieval-augmented generati

In [28]:
#testing with a question outside the context
unknown_question = "Who invented the telephone?"
print(f"Question: {unknown_question}")
print(f"Answer: {rag_chain.invoke(unknown_question)}")

Question: Who invented the telephone?
Answer: I don't know.
